# FairFace IAT Comparison

This notebook evaluates the base model plus one or more LoRA adapters on the same FairFace sample and produces the same word-count plot style as `iat_plots_share_with_URAP.ipynb`, using the base model output as the shared word-order reference.


In [ ]:
from pathlib import Path
from collections import Counter
import gc
import importlib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time

ROOT_CANDIDATES = [
    Path('/workspace/fairface_latest_code'),
    Path.cwd(),
    Path.cwd() / 'fairface_latest_code',
]
ROOT = next((p for p in ROOT_CANDIDATES if p.exists() and p.name == 'fairface_latest_code'), ROOT_CANDIDATES[0])
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = ROOT / 'outputs'
LORA_WEIGHTS_DIR = ROOT / 'lora_weights'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
LORA_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_SCRIPT_CANDIDATES = [
    ROOT.parent / 'analysis' / 'train_fairface_iat_dpo.py',
    Path('/workspace/analysis/train_fairface_iat_dpo.py'),
    ROOT / 'train_fairface_iat_dpo_latest.py',
]
SOURCE_SCRIPT = next((p for p in SOURCE_SCRIPT_CANDIDATES if p.exists()), None)
if SOURCE_SCRIPT is None:
    raise FileNotFoundError('Could not find a FairFace trainer snapshot or analysis/train_fairface_iat_dpo.py')
SNAPSHOT_SCRIPT = ROOT / 'train_fairface_iat_dpo_latest.py'
if SOURCE_SCRIPT != SNAPSHOT_SCRIPT:
    shutil.copy2(SOURCE_SCRIPT, SNAPSHOT_SCRIPT)
print('ROOT:', ROOT)
print('SNAPSHOT_SCRIPT:', SNAPSHOT_SCRIPT)


In [ ]:
REQUIRED = {
    'packaging': 'packaging>=24.2',
    'pandas': 'pandas==2.2.3',
    'PIL': 'pillow==11.1.0',
    'matplotlib': 'matplotlib==3.10.0',
    'transformers': 'transformers==4.50.0',
    'accelerate': 'accelerate==1.4.0',
    'peft': 'peft==0.14.0',
    'huggingface_hub': 'huggingface_hub==0.28.1',
    'datasets': 'datasets>=3.4.0',
    'sentencepiece': 'sentencepiece==0.2.0',
}

missing = []
for module_name, pip_spec in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pip_spec)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display


In [ ]:
spec = importlib.util.spec_from_file_location('fairface_latest_train', SNAPSHOT_SCRIPT)
fairface = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(fairface)

index_path = ROOT / 'run_index.csv'
if index_path.exists():
    display(pd.read_csv(index_path).sort_values('created_at', ascending=False).head(20))

print('Available adapter folders:')
for path in sorted(LORA_WEIGHTS_DIR.iterdir()):
    if path.is_dir():
        print('-', path.name)


In [ ]:
# Comparison configuration
RUN_LABEL = 'compare'
DATASET_NAME = 'HuggingFaceM4/FairFace'
DATASET_CONFIG = '0.25'
EVAL_SPLIT = 'validation'
EVAL_SAMPLE_PER_GROUP = 24
SEED = 17
EVAL_SEED = 1017
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_NEW_TOKENS = 30
FALLBACK_MAX_NEW_TOKENS = 20
EVAL_REPEATS = 10
REUSE_EVAL_METADATA_CSV = None

# If REUSE_EVAL_METADATA_CSV is None, this notebook generates one fresh shared eval sample
# and reuses that same sample for the base model and every adapter in this comparison run.

# Adapter selection mode: 'latest' or 'manual'
ADAPTER_SELECTION_MODE = 'latest'

# When using latest mode, pick the newest N adapter folders from lora_weights/.
LATEST_ADAPTER_COUNT = 1

# When using manual mode, list one or more adapters here. Paths can be relative to ROOT or absolute.
MANUAL_ADAPTER_SPECS = [
    # {'label': 'seed17_effect', 'path': ROOT / 'lora_weights' / 'fairface_dpo_default_20260429_120000'},
]


In [ ]:
POS_WORDS = fairface.POS_WORDS
NEG_WORDS = fairface.NEG_WORDS
WORDS_LIST = POS_WORDS + NEG_WORDS
WORDS_SET = set(WORDS_LIST)

def plot_word_counts_with_reference(
    csv_path: str,
    out_png_path: str,
    reference_path: str | None = None,
    image_col_in_csv: str = 'image',
    response_col_in_csv: str = 'response',
    image_col_std: str = 'Image File',
    response_col_std: str = 'Response Text',
    figsize_base: tuple = (12, 8),
    dpi: int = 300,
    print_out_of_list: bool = True,
    title_fs: int = 22,
    label_fs: int = 18,
    tick_fs: int = 13,
    legend_fs: int = 16,
):
    def standardize_cols(d: pd.DataFrame) -> pd.DataFrame:
        d = d.copy()
        rename = {}
        if image_col_std not in d.columns:
            for c in [image_col_in_csv, 'image_file', 'image', 'Image', 'file', 'filename', 'Image File']:
                if c in d.columns:
                    rename[c] = image_col_std
                    break
        if response_col_std not in d.columns:
            for c in [response_col_in_csv, 'response', 'Response', 'resp', 'output', 'text', 'Response Text']:
                if c in d.columns:
                    rename[c] = response_col_std
                    break
        if rename:
            d = d.rename(columns=rename)
        missing = [c for c in [image_col_std, response_col_std] if c not in d.columns]
        if missing:
            raise ValueError(f'Missing required columns {missing}. Available columns: {list(d.columns)}')
        return d

    df = standardize_cols(pd.read_csv(csv_path))
    ref = df if not reference_path else standardize_cols(pd.read_csv(reference_path))

    def infer_bw_prefix(image_file: str):
        s = os.path.basename(str(image_file)).lower().strip()
        if s.startswith('bm') or 'black' in s:
            return 'b'
        if s.startswith('wm') or 'white' in s:
            return 'w'
        if s.startswith('b'):
            return 'b'
        if s.startswith('w'):
            return 'w'
        if re.match(r'^\d+b', s):
            return 'b'
        if re.match(r'^\d+w', s):
            return 'w'
        return None

    def parse_words(text: str):
        s = str(text).replace('.', '')
        parts = re.split(r'[,\n;]+', s)
        return [p.strip() for p in parts if p.strip()]

    def count_counters(data: pd.DataFrame):
        b_counter = Counter()
        w_counter = Counter()
        out_of_list_counter = Counter()
        for _, row in data.iterrows():
            pref = infer_bw_prefix(row[image_col_std])
            for word in parse_words(row[response_col_std]):
                if word in WORDS_SET:
                    if pref == 'b':
                        b_counter[word] += 1
                    elif pref == 'w':
                        w_counter[word] += 1
                else:
                    out_of_list_counter[word] += 1
        return b_counter, w_counter, out_of_list_counter

    b_counter, w_counter, out_of_list_counter = count_counters(df)
    ref_b, ref_w, _ = count_counters(ref)

    def is_nonzero_ref(word: str) -> bool:
        return (ref_b.get(word, 0) + ref_w.get(word, 0)) > 0

    pos_nonzero = [w for w in POS_WORDS if is_nonzero_ref(w)]
    pos_zero = [w for w in POS_WORDS if not is_nonzero_ref(w)]
    pos_nonzero_sorted = sorted(pos_nonzero, key=lambda w: (-ref_w.get(w, 0), w))
    neg_nonzero = [w for w in NEG_WORDS if is_nonzero_ref(w)]
    neg_zero = [w for w in NEG_WORDS if not is_nonzero_ref(w)]
    neg_nonzero_sorted = sorted(neg_nonzero, key=lambda w: (-ref_b.get(w, 0), w))
    order_words = pos_nonzero_sorted + pos_zero + neg_nonzero_sorted + neg_zero

    count_df = pd.DataFrame({
        'Word': order_words,
        'b_count': [b_counter.get(w, 0) for w in order_words],
        'w_count': [w_counter.get(w, 0) for w in order_words],
    })

    if print_out_of_list:
        if out_of_list_counter:
            tmp = pd.DataFrame(out_of_list_counter.items(), columns=['Word', 'Count']).sort_values('Count', ascending=False)
            display(tmp)
        else:
            print('All generated words were in the list.')

    width = max(figsize_base[0], 0.22 * len(count_df))
    fig, ax = plt.subplots(figsize=(width, figsize_base[1]), dpi=dpi)
    bar_width = 0.35
    x = range(len(count_df))
    ax.bar(x, count_df['b_count'], bar_width, label='black photo', color='#0072B2')
    ax.bar([i + bar_width for i in x], count_df['w_count'], bar_width, label='white photo', color='#E69F00')
    ax.set_xlabel('Words', fontsize=label_fs)
    ax.set_ylabel('Counts', fontsize=label_fs)
    ax.set_title('Word Counts by Image File Prefix', fontsize=title_fs)
    ax.set_xticks([i + bar_width / 2 for i in x])
    ax.set_xticklabels(count_df['Word'], rotation=90, fontsize=tick_fs)
    ax.tick_params(axis='y', labelsize=tick_fs)
    ax.legend(fontsize=legend_fs)
    plt.tight_layout()
    plt.savefig(out_png_path, bbox_inches='tight')
    plt.close(fig)
    return count_df, order_words, out_of_list_counter


In [ ]:
def slugify(value: str) -> str:
    value = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value).strip())
    return value.strip('._-') or 'run'

def make_unique_run_name(prefix: str, label: str) -> str:
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    base = f"{prefix}_{slugify(label)}_{timestamp}" if label else f"{prefix}_{timestamp}"
    candidate = base
    counter = 2
    while (OUTPUTS_DIR / candidate).exists():
        candidate = f"{base}_{counter:02d}"
        counter += 1
    return candidate

def normalize_adapter_specs(specs):
    normalized = []
    for item in specs:
        if isinstance(item, dict):
            label = item.get('label') or Path(item['path']).name
            path = Path(item['path'])
        else:
            path = Path(item)
            label = path.name
        if not path.is_absolute():
            path = (ROOT / path).resolve()
        normalized.append({'label': label, 'path': path})
    return normalized

def list_available_adapter_dirs():
    return sorted(
        [path for path in LORA_WEIGHTS_DIR.iterdir() if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

def resolve_adapter_specs(selection_mode, manual_specs, latest_count: int):
    mode = str(selection_mode).strip().lower()
    if mode == 'latest':
        available = list_available_adapter_dirs()
        if not available:
            raise FileNotFoundError(f'No adapter folders found under {LORA_WEIGHTS_DIR}')
        count = max(1, int(latest_count))
        return [{'label': path.name, 'path': path.resolve()} for path in available[:count]]
    if mode == 'manual':
        if not manual_specs:
            raise ValueError('MANUAL_ADAPTER_SPECS is empty while ADAPTER_SELECTION_MODE is manual')
        return normalize_adapter_specs(manual_specs)
    raise ValueError("ADAPTER_SELECTION_MODE must be 'latest' or 'manual'")

def run_eval_for_model(label: str, adapter_dir: Path | None, eval_rows, base_model_source_ref: str, base_model_source_is_local: bool, processor):
    model = fairface.load_multimodal_model(base_model_source_ref, base_model_source_is_local)
    if adapter_dir is not None:
        model = fairface.PeftModel.from_pretrained(model, str(adapter_dir))
    model.eval()
    pipe = fairface.build_pipe_from_model(model, processor)
    df = fairface.run_eval_condition(
        pipe,
        eval_rows,
        repeats=EVAL_REPEATS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS,
        fallback_max_new_tokens=FALLBACK_MAX_NEW_TOKENS,
    )
    del pipe
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return df

fairface.ensure_sentencepiece()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

adapter_specs = resolve_adapter_specs(ADAPTER_SELECTION_MODE, MANUAL_ADAPTER_SPECS, LATEST_ADAPTER_COUNT)
print('Selected adapters:')
for spec_item in adapter_specs:
    print('-', spec_item['label'], '->', spec_item['path'])
for spec_item in adapter_specs:
    if not spec_item['path'].exists():
        raise FileNotFoundError(f"Adapter path not found: {spec_item['path']}")

RUN_NAME = make_unique_run_name('fairface_compare', RUN_LABEL)
RUN_DIR = OUTPUTS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
LOGICAL_EVAL_SEED = int(EVAL_SEED if EVAL_SEED is not None else SEED + 1000)
effective_eval_metadata_csv = Path(REUSE_EVAL_METADATA_CSV) if REUSE_EVAL_METADATA_CSV else None
if effective_eval_metadata_csv is not None and not effective_eval_metadata_csv.is_absolute():
    effective_eval_metadata_csv = (ROOT / effective_eval_metadata_csv).resolve()
if effective_eval_metadata_csv is not None:
    print('Using eval metadata:', effective_eval_metadata_csv)
else:
    print('Generating one fresh shared eval sample with EVAL_SAMPLE_PER_GROUP =', EVAL_SAMPLE_PER_GROUP, 'and EVAL_SEED =', LOGICAL_EVAL_SEED)
eval_root = RUN_DIR / 'eval_sample'
eval_ds = fairface.load_dataset(DATASET_NAME, DATASET_CONFIG, split=EVAL_SPLIT)
if effective_eval_metadata_csv is not None:
    eval_rows, eval_meta_path = fairface.materialize_dataset_sample_from_metadata(eval_ds, 'eval', effective_eval_metadata_csv, eval_root)
else:
    eval_rows, eval_meta_path = fairface.save_dataset_sample(eval_ds, 'eval', EVAL_SAMPLE_PER_GROUP, eval_root, LOGICAL_EVAL_SEED)

base_model_source, base_model_source_kind = fairface.resolve_base_model_source(fairface.BASE_MODEL_ID, os.environ.get('BASE_MODEL_LOCAL_DIR', os.environ.get('GEMMA_LOCAL_DIR', '')).strip())
base_model_source_ref = str(base_model_source)
base_model_source_is_local = base_model_source_kind != 'hf_repo'
processor, processor_use_fast = fairface.load_processor(base_model_source_ref, base_model_source_is_local)

results = []
reference_csv_path = None
base_df = run_eval_for_model('base', None, eval_rows, base_model_source_ref, base_model_source_is_local, processor)
base_csv_path = RUN_DIR / 'base.csv'
base_df.to_csv(base_csv_path, index=False)
reference_csv_path = base_csv_path
base_png_path = RUN_DIR / 'base.png'
base_count_df, _, _ = plot_word_counts_with_reference(str(base_csv_path), str(base_png_path), reference_path=str(reference_csv_path), title_fs=26, label_fs=20, tick_fs=14, legend_fs=18)
base_metrics = fairface.compute_eval_metrics_from_counts(base_count_df)
results.append({'label': 'base', 'adapter_dir': '', 'csv': str(base_csv_path), 'png': str(base_png_path), **base_metrics})

for spec_item in adapter_specs:
    safe_label = slugify(spec_item['label'])
    df = run_eval_for_model(spec_item['label'], spec_item['path'], eval_rows, base_model_source_ref, base_model_source_is_local, processor)
    csv_path = RUN_DIR / f'{safe_label}.csv'
    df.to_csv(csv_path, index=False)
    png_path = RUN_DIR / f'{safe_label}.png'
    count_df, _, _ = plot_word_counts_with_reference(str(csv_path), str(png_path), reference_path=str(reference_csv_path), title_fs=26, label_fs=20, tick_fs=14, legend_fs=18)
    metrics = fairface.compute_eval_metrics_from_counts(count_df)
    results.append({'label': spec_item['label'], 'adapter_dir': str(spec_item['path']), 'csv': str(csv_path), 'png': str(png_path), **metrics})

summary_df = pd.DataFrame(results)
summary_path = RUN_DIR / 'comparison_metrics.csv'
summary_df.to_csv(summary_path, index=False)
manifest = {
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR),
    'eval_metadata_csv': str(eval_meta_path),
    'dataset_name': DATASET_NAME,
    'dataset_config': DATASET_CONFIG,
    'eval_split': EVAL_SPLIT,
    'eval_sample_per_group': int(EVAL_SAMPLE_PER_GROUP),
    'eval_repeats': int(EVAL_REPEATS),
    'seed': int(SEED),
    'eval_seed': int(LOGICAL_EVAL_SEED),
    'effective_eval_metadata_csv': str(eval_meta_path),
    'base_model_source': base_model_source_ref,
    'base_model_source_kind': base_model_source_kind,
    'processor_use_fast': processor_use_fast,
    'adapters': [{'label': item['label'], 'path': str(item['path'])} for item in adapter_specs],
    'summary_csv': str(summary_path),
}
(RUN_DIR / 'comparison_manifest.json').write_text(json.dumps(manifest, indent=2))
display(summary_df)
for row in results:
    display(Markdown(f"### {row['label']}: {Path(row['png']).name}"))
    display(Image(filename=row['png']))

print('Comparison outputs written to:', RUN_DIR)
